# Задача 1. Tracer

**Объяснение решения**

Текущую глубину вызовов храним в `_depth`. Начальная глубина равна `0`.

При каждом вызове выполняем следующие действия:

1. Получаем текущую глубину и печатаем имя функции с аргументами. + еще добавляем 2 пробела.
2. Увеличиваем глубину на 1 и вызываем исходную функцию
3. Когда функция возвращает результат, печатаем его с отступом этого вызова и возвращаем тот же результат вызывающему коду.
4. В блоке finally восстанавливаем прежнюю глубину

Для входа используем `->`, для возврата - `<-`. Если функция выбросила исключение, показываем его с меткой `!!` и передаём дальше через `raise`.

**Решение**

In [1]:
from contextvars import ContextVar
from functools import wraps


_depth = ContextVar("tracer_depth", default=0)


def _format_arguments(args, kwargs):
    arguments = [repr(value) for value in args]
    arguments.extend(f"{name}={value!r}" for name, value in kwargs.items())
    return ", ".join(arguments)


def tracer(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        depth = _depth.get()
        print(f"{'  ' * depth}-> {function.__name__}({_format_arguments(args, kwargs)})")
        token = _depth.set(depth + 1)

        try:
            result = function(*args, **kwargs)
        except Exception as error:
            print(f"{'  ' * depth}!! {function.__name__}: {error!r}")
            raise
        else:
            print(f"{'  ' * depth}<- {function.__name__} = {result!r}")
            return result
        finally:
            _depth.reset(token)

    return wrapper

**Пример**

Посмотрим на факториал, как рассматривали на паре. Базовый случай - `factorial(0)`, который возвращает `1`. Каждый следующий вызов умножает полученный
результат на своё значение `n`.

In [2]:
@tracer
def factorial(n):
    if n < 0:
        raise ValueError("n должно быть неотрицательным")
    if n == 0:
        return 1
    return n * factorial(n - 1)


result = factorial(3)
assert result == 6

-> factorial(3)
  -> factorial(2)
    -> factorial(1)
      -> factorial(0)
      <- factorial = 1
    <- factorial = 1
  <- factorial = 2
<- factorial = 6


**Пошаговая работа декоратора.**

| Шаг | Событие | Глубина | Отступ | Результат |
|---|---|---:|---|---|
| 1 | Вход в `factorial(3)` | 0 | 0 пробелов | — |
| 2 | Вход в `factorial(2)` | 1 | 2 пробела | — |
| 3 | Вход в `factorial(1)` | 2 | 4 пробела | — |
| 4 | Вход в `factorial(0)` | 3 | 6 пробелов | — |
| 5 | Возврат из `factorial(0)` | 3 | 6 пробелов | `1` |
| 6 | Возврат из `factorial(1)` | 2 | 4 пробела | `1` |
| 7 | Возврат из `factorial(2)` | 1 | 2 пробела | `2` |
| 8 | Возврат из `factorial(3)` | 0 | 0 пробелов | `6` |

Еще рассмотрим `fibonacci(3)` тоже с пары.

In [3]:
@tracer
def fibonacci(n):
    if n < 0:
        raise ValueError("n должно быть неотрицательным")
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)


result = fibonacci(3)
assert result == 2

-> fibonacci(3)
  -> fibonacci(2)
    -> fibonacci(1)
    <- fibonacci = 1
    -> fibonacci(0)
    <- fibonacci = 0
  <- fibonacci = 1
  -> fibonacci(1)
  <- fibonacci = 1
<- fibonacci = 2


**Сложность**

Пусть `N` - общее число вызовов функции, `h` - максимальная глубина стека, а `S` - суммарное количество напечатанных символов, включая отступы.
Здесь `L` - максимальная длина формируемого сообщения.

| Что учитываем | Время | Дополнительная память |
|---|---|---|
| Управление глубиной и вызовами | `O(N)` | `O(h)` |
| Трассировку с формированием строк и печатью | `O(N + S)` | `O(h + L)` |

Во время работы храним состояние для активных вызовов и временно формируем
сообщение для вывода.

**Тесты**

In [3]:
import inspect
import unittest
from contextlib import redirect_stdout
from io import StringIO


class TestTracer(unittest.TestCase):
    def capture(self, function, *args, **kwargs):
        output = StringIO()
        with redirect_stdout(output):
            result = function(*args, **kwargs)
        return result, output.getvalue().splitlines()

    def test_factorial(self):
        @tracer
        def factorial(n):
            if n == 0:
                return 1
            return n * factorial(n - 1)

        result, lines = self.capture(factorial, 3)
        self.assertEqual(result, 6)
        self.assertEqual(lines, [
            "-> factorial(3)",
            "  -> factorial(2)",
            "    -> factorial(1)",
            "      -> factorial(0)",
            "      <- factorial = 1",
            "    <- factorial = 1",
            "  <- factorial = 2",
            "<- factorial = 6",
        ])

    def test_base_case(self):
        @tracer
        def count(n):
            if n == 0:
                return 0
            return 1 + count(n - 1)

        result, lines = self.capture(count, 0)
        self.assertEqual(result, 0)
        self.assertEqual(lines, ["-> count(0)", "<- count = 0"])

    def test_branching_recursion(self):
        @tracer
        def fibonacci(n):
            if n <= 1:
                return n
            return fibonacci(n - 1) + fibonacci(n - 2)

        result, lines = self.capture(fibonacci, 3)
        self.assertEqual(result, 2)
        self.assertEqual(lines, [
            "-> fibonacci(3)",
            "  -> fibonacci(2)",
            "    -> fibonacci(1)",
            "    <- fibonacci = 1",
            "    -> fibonacci(0)",
            "    <- fibonacci = 0",
            "  <- fibonacci = 1",
            "  -> fibonacci(1)",
            "  <- fibonacci = 1",
            "<- fibonacci = 2",
        ])

    def test_no_arguments(self):
        @tracer
        def answer():
            return 42

        result, lines = self.capture(answer)
        self.assertEqual(result, 42)
        self.assertEqual(lines, ["-> answer()", "<- answer = 42"])

    def test_args_and_kwargs(self):
        @tracer
        def combine(a, *values, scale=1, **options):
            return (a + sum(values)) * scale + options.get("offset", 0)

        result, lines = self.capture(combine, 2, 3, 4, scale=2, offset=1)
        self.assertEqual(result, 19)
        self.assertEqual(lines, [
            "-> combine(2, 3, 4, scale=2, offset=1)",
            "<- combine = 19",
        ])

    def test_keyword_recursion(self):
        @tracer
        def total(*, n):
            if n == 0:
                return 0
            return n + total(n=n - 1)

        result, lines = self.capture(total, n=2)
        self.assertEqual(result, 3)
        self.assertEqual(lines, [
            "-> total(n=2)",
            "  -> total(n=1)",
            "    -> total(n=0)",
            "    <- total = 0",
            "  <- total = 1",
            "<- total = 3",
        ])

    def test_none_result(self):
        @tracer
        def nothing():
            return None

        result, lines = self.capture(nothing)
        self.assertIsNone(result)
        self.assertEqual(lines, ["-> nothing()", "<- nothing = None"])

    def test_string_representation(self):
        @tracer
        def echo(value):
            return value

        value = "hello\n'world'"
        result, lines = self.capture(echo, value)
        self.assertEqual(result, value)
        self.assertEqual(lines, [
            f"-> echo({value!r})",
            f"<- echo = {value!r}",
        ])

    def test_repeated_calls(self):
        @tracer
        def count(n):
            if n == 0:
                return 0
            return 1 + count(n - 1)

        self.capture(count, 3)
        result, lines = self.capture(count, 1)
        self.assertEqual(result, 1)
        self.assertEqual(lines, [
            "-> count(1)",
            "  -> count(0)",
            "  <- count = 0",
            "<- count = 1",
        ])

    def test_nested_functions(self):
        @tracer
        def twice(value):
            return value * 2

        @tracer
        def outer(value):
            return twice(value) + 1

        result, lines = self.capture(outer, 2)
        self.assertEqual(result, 5)
        self.assertEqual(lines, [
            "-> outer(2)",
            "  -> twice(2)",
            "  <- twice = 4",
            "<- outer = 5",
        ])

    def test_mutual_recursion(self):
        @tracer
        def is_even(n):
            return True if n == 0 else is_odd(n - 1)

        @tracer
        def is_odd(n):
            return False if n == 0 else is_even(n - 1)

        result, lines = self.capture(is_even, 3)
        self.assertIs(result, False)
        self.assertEqual(lines, [
            "-> is_even(3)",
            "  -> is_odd(2)",
            "    -> is_even(1)",
            "      -> is_odd(0)",
            "      <- is_odd = False",
            "    <- is_even = False",
            "  <- is_odd = False",
            "<- is_even = False",
        ])

    def test_exception_restores_depth(self):
        error = ValueError("boom")

        @tracer
        def fail(n):
            if n == 0:
                raise error
            return fail(n - 1)

        @tracer
        def ok():
            return 7

        output = StringIO()
        with redirect_stdout(output):
            with self.assertRaises(ValueError) as caught:
                fail(2)
            self.assertEqual(ok(), 7)

        self.assertIs(caught.exception, error)
        self.assertEqual(output.getvalue().splitlines(), [
            "-> fail(2)",
            "  -> fail(1)",
            "    -> fail(0)",
            "    !! fail: ValueError('boom')",
            "  !! fail: ValueError('boom')",
            "!! fail: ValueError('boom')",
            "-> ok()",
            "<- ok = 7",
        ])

    def test_handled_exception(self):
        @tracer
        def broken():
            raise ValueError("boom")

        @tracer
        def ok():
            return 9

        @tracer
        def recover():
            try:
                broken()
            except ValueError:
                return ok()

        result, lines = self.capture(recover)
        self.assertEqual(result, 9)
        self.assertEqual(lines, [
            "-> recover()",
            "  -> broken()",
            "  !! broken: ValueError('boom')",
            "  -> ok()",
            "  <- ok = 9",
            "<- recover = 9",
        ])

    def test_metadata(self):
        def add(a: int, b: int = 2) -> int:
            """Складывает два числа."""
            return a + b

        decorated = tracer(add)
        self.assertEqual(decorated.__name__, add.__name__)
        self.assertEqual(decorated.__doc__, add.__doc__)
        self.assertIs(decorated.__wrapped__, add)
        self.assertEqual(inspect.signature(decorated), inspect.signature(add))

    def test_original_result_and_single_call(self):
        expected = [1, 2]
        calls = []

        @tracer
        def make():
            calls.append(1)
            return expected

        result, lines = self.capture(make)
        self.assertIs(result, expected)
        self.assertEqual(calls, [1])
        self.assertEqual(lines, ["-> make()", "<- make = [1, 2]"])

In [4]:
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestTracer)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful(), "Некоторые тесты завершились ошибкой"
print(f"Успешно пройдено тестов: {test_result.testsRun}")

test_args_and_kwargs (__main__.TestTracer.test_args_and_kwargs) ... ok
test_base_case (__main__.TestTracer.test_base_case) ... ok
test_branching_recursion (__main__.TestTracer.test_branching_recursion) ... ok
test_exception_restores_depth (__main__.TestTracer.test_exception_restores_depth) ... ok
test_factorial (__main__.TestTracer.test_factorial) ... ok
test_handled_exception (__main__.TestTracer.test_handled_exception) ... ok
test_keyword_recursion (__main__.TestTracer.test_keyword_recursion) ... ok
test_metadata (__main__.TestTracer.test_metadata) ... ok
test_mutual_recursion (__main__.TestTracer.test_mutual_recursion) ... ok
test_nested_functions (__main__.TestTracer.test_nested_functions) ... ok
test_no_arguments (__main__.TestTracer.test_no_arguments) ... ok
test_none_result (__main__.TestTracer.test_none_result) ... ok
test_original_result_and_single_call (__main__.TestTracer.test_original_result_and_single_call) ... ok
test_repeated_calls (__main__.TestTracer.test_repeated_call

Успешно пройдено тестов: 15
